# Total Variation — Motion Deblurring and Denoising

Questo notebook usa la regolarizzazione Total Variation come metodo baseline per il problema di motion deblurring e denoising.

La degradazione è la stessa usata per NAFNet:

- motion blur con kernel `9x9`;
- angolo del blur pari a `45°`;
- rumore gaussiano con livelli `[0.005, 0.01, 0.05, 0.1]`.

Per ogni immagine e per ogni livello di rumore vengono testati diversi valori di `λ`.  
La ricostruzione migliore viene scelta in base al PSNR rispetto all'immagine pulita.

In [1]:
# 1 - Setup and dataset

import os
from datasets import load_dataset
import torch
from torch.utils.data import Subset

from IPPy.utilities.metrics import PSNR, SSIM, RE

from image_dataset import ImageDataset
from degradation import ImageDegradation, DegradationParameters
from utils import plot  # type: ignore
from config import TEST_SIZE, TO_RECONSTRUCT_INDEXES
from models import TotalVariationRegularizer


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 256
NOISE_LEVELS = [0.005, 0.01, 0.05, 0.1]

RESULTS_DIR = "results/TV"

LAMBDA_VALUES = [
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
    3e-1,
]


ds = load_dataset("benjamin-paine/imagenet-1k-256x256")

test_dataset = Subset(
    ImageDataset(ds["test"]),
    range(TEST_SIZE),
)


tv = TotalVariationRegularizer(
    lambda_values=LAMBDA_VALUES,
    max_iters=100,
)

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

## Reconstruction

Per ogni immagine selezionata vengono generate quattro versioni degradate, una per ciascun livello di rumore.

Per ogni degradazione, il metodo TV ricostruisce l'immagine usando tutti i valori di `λ` definiti in `LAMBDA_VALUES`.  
La migliore ricostruzione viene selezionata in base al PSNR.

In [ ]:
# 2 - TV reconstruction on selected test images

all_tv_results = []

for image_index in TO_RECONSTRUCT_INDEXES:
    clean = test_dataset[image_index].unsqueeze(0).to(DEVICE)

    for noise_level in NOISE_LEVELS:
        noise_name = f"noise_{int(noise_level * 1000):03d}"

        degradation = ImageDegradation(
            DegradationParameters(
                image_size=IMAGE_SIZE,
                kernel_type="motion",
                kernel_size=9,
                motion_angle=45,
                noise_levels=[noise_level],
            )
        )

        degraded = degradation(clean)

        save_dir = os.path.join(
            RESULTS_DIR,
            noise_name,
            f"image_{image_index}",
            "lambdas",
        )

        results = tv(
            y_d=degraded,
            K=degradation.operator,
            x_gt=clean,
            save_dir=save_dir,
            preview=False,
        )

        best_result = max(
            results,
            key=lambda result: result["psnr"],
        )

        best_reconstruction = best_result["reconstruction"]

        psnr_degraded = PSNR(degraded, clean)
        ssim_degraded = SSIM(degraded, clean)

        summary = {
            "image_index": image_index,
            "noise_level": noise_level,
            "best_lambda": best_result["lambda"],
            "psnr_degraded": psnr_degraded,
            "ssim_degraded": ssim_degraded,
            "psnr_tv": best_result["psnr"],
            "ssim_tv": best_result["ssim"],
            "re_tv": best_result["re"],
        }

        all_tv_results.append(summary)

        plot(
            clean,
            degraded,
            best_reconstruction,
            titles=[
                f"Clean\nindex: {image_index}",
                (
                    f"Degraded\n"
                    f"noise: {noise_level}\n"
                    f"PSNR: {psnr_degraded:.2f} dB\n"
                    f"SSIM: {ssim_degraded:.4f}"
                ),
                (
                    f"TV reconstruction\n"
                    f"λ: {best_result['lambda']:.1e}\n"
                    f"PSNR: {best_result['psnr']:.2f} dB\n"
                    f"SSIM: {best_result['ssim']:.4f}"
                ),
            ],
            size=4,
            save_path=os.path.join(
                RESULTS_DIR,
                noise_name,
                f"image_{image_index}.png",
            ),
        )

Running TV reconstruction with lambda = 1.0e-04


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## Results summary

La cella seguente stampa una tabella compatta con il miglior valore di `λ` trovato per ogni immagine e per ogni livello di rumore.

In [ ]:
# 3 - Results summary

for result in all_tv_results:
    print(
        f"image={result['image_index']} | "
        f"noise={result['noise_level']} | "
        f"best λ={result['best_lambda']:.1e} | "
        f"degraded PSNR={result['psnr_degraded']:.2f} dB | "
        f"TV PSNR={result['psnr_tv']:.2f} dB | "
        f"TV SSIM={result['ssim_tv']:.4f} | "
        f"TV RE={result['re_tv']:.4f}"
    )